# آموزش MoE روی Kaggle GPU: متن ChatGPT + logits معلم Qwen

این نوت‌بوک همان مسیر واحد ترکیبی را اجرا می‌کند: پاسخ‌های متنی برای Cross-Entropy به‌علاوهٔ تقطیر sparse logits مدل Qwen در همان حلقهٔ آموزش. مسیرهای text-only و logits-only اجرا نمی‌شوند.

**هشدار:** آموزش کامل Qwen3.5-2B می‌تواند چند ساعت طول بکشد و به سهمیه و دسترس‌پذیری GPU در Kaggle وابسته است. همهٔ خروجی‌ها زیر /kaggle/working/qwen-moe-chatgpt-logits ذخیره می‌شوند و فایل‌های خروجی Notebook پس از پایان اجرا در بخش Output قرار می‌گیرند. این pipeline به‌تنهایی مدل مستقل Android نمی‌سازد؛ در پایان، checkpointها و گزارش ارزیابی ذخیره می‌شوند.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os, hashlib, shutil, zipfile

REPO = Path('/kaggle/working/Qwen-MoE-0.4B-')
WORK = Path('/kaggle/working/qwen-moe-chatgpt-logits')
SOURCE_JSONL = REPO / 'data/chatgpt_prompt_response_archive_part2.jsonl'
INPUT_JSONL = SOURCE_JSONL
TRAIN_JSONL = WORK / 'train_archive.jsonl'
EVAL_JSONL = WORK / 'eval_holdout.jsonl'
EVAL_TEXT = WORK / 'eval_holdout.txt'
LOGITS_OUT = WORK / 'qwen_logits_output_v2'
INITIAL_CHECKPOINTS = WORK / 'analysis_initialized_checkpoints_v2'
ANALYSIS_DIR = WORK / 'architecture_analysis_v2'
ANALYSIS_TEXT = WORK / 'analysis_corpus.txt'
EXPERT_PLAN = ANALYSIS_DIR / 'expert_channel_plan.json'
TRAIN_OUT = WORK / 'combined_moe_training_v2'
MODEL_ID = 'Qwen/Qwen3.5-2B'
TOP_K = 64
TEMPERATURE = 1.0
MAX_SEQ_LENGTH = 256

REPO_URL = 'https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git'
if not REPO.exists():
    # Retry transient DNS/network failures before falling back to the source ZIP.
    import time
    clone = None
    for attempt in range(1, 4):
        clone = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)],
            text=True, capture_output=True
        )
        if clone.returncode == 0:
            break
        print(f'git clone attempt {attempt}/3 failed: {clone.stderr[-700:]}')
        if REPO.exists():
            shutil.rmtree(REPO, ignore_errors=True)
        if attempt < 3:
            time.sleep(8 * attempt)
    if clone is None or clone.returncode != 0:
        print('Git clone retries failed; trying GitHub source ZIP with retries.')
        import urllib.request
        import io
        archive_url = 'https://codeload.github.com/samanramezani1377-hub/Qwen-MoE-0.4B-/zip/refs/heads/main'
        zip_error = None
        for attempt in range(1, 4):
            try:
                with urllib.request.urlopen(archive_url, timeout=90) as response:
                    archive_bytes = response.read()
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    archive.extractall(REPO.parent)
                extracted = next(p for p in REPO.parent.glob('Qwen-MoE-0.4B-*') if p.is_dir() and (p / 'data').exists())
                if REPO.exists():
                    shutil.rmtree(REPO, ignore_errors=True)
                extracted.rename(REPO)
                zip_error = None
                break
            except Exception as exc:
                zip_error = exc
                print(f'GitHub ZIP attempt {attempt}/3 failed: {exc!r}')
                if attempt < 3:
                    time.sleep(8 * attempt)
        if zip_error is not None:
            raise RuntimeError(
                'GitHub remained unreachable after retries (git + source ZIP). '
                'The Kaggle runtime has a DNS/network connectivity problem; '
                'confirm Notebook Settings > Internet is enabled, then restart '
                'the session or rerun the Kaggle kernel later. Internet is also '
                'required to install Transformers and download the Qwen model. '
                'Original git error: ' + (clone.stderr[-900:] if clone else 'no git result')
                + ' | Last ZIP error: ' + repr(zip_error)
            ) from zip_error
else:
    pull = subprocess.run(
        ['git', '-C', str(REPO), 'pull', '--ff-only'],
        text=True, capture_output=True
    )
    if pull.returncode != 0:
        print('git pull failed; continuing with the already-present repository copy.')
        print(pull.stderr[-2000:])

if not __import__('torch').cuda.is_available():
    raise RuntimeError('Kaggle GPU is not enabled/available. Edit notebook settings and enable GPU, then rerun.')

print('GPU:', __import__('torch').cuda.get_device_name(0))
print('Working directory:', WORK)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'git+https://github.com/huggingface/transformers.git@main',
                'accelerate', 'safetensors', 'torchvision', 'pillow'], check=True)
from transformers import AutoModelForMultimodalLM
print('Transformers main branch and Qwen3.5 loader are available.')

if not SOURCE_JSONL.is_file():
    raise FileNotFoundError(f'ChatGPT text archive not found: {SOURCE_JSONL}')
WORK.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print('Input archive:', SOURCE_JSONL)
print('Pipeline: ChatGPT response CE + Qwen sparse-logit KD only')

In [ ]:
# در محیط Kaggle هیچ فایل checkpoint قدیمی خارج از پوشهٔ کاری دست‌کاری یا حذف نمی‌شود.

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
import hashlib
all_rows = [json.loads(line) for line in SOURCE_JSONL.read_text(encoding='utf-8').splitlines() if line.strip()]
train_rows, eval_rows = [], []
for row in all_rows:
    sid = str(row.get('sample_id',''))
    bucket = int(hashlib.sha256(sid.encode('utf-8')).hexdigest()[:8], 16) % 10
    (eval_rows if bucket == 0 else train_rows).append(row)
if len(eval_rows) < 2 or len(train_rows) < 4:
    raise RuntimeError(f'Invalid stable split: train={len(train_rows)}, eval={len(eval_rows)}')
TRAIN_JSONL.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in train_rows), encoding='utf-8')
EVAL_JSONL.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in eval_rows), encoding='utf-8')
INPUT_JSONL = TRAIN_JSONL
ANALYSIS_TEXT.write_text('\n'.join(
    (str(row.get('prompt','')).strip() + ' ' + str(row.get('chatgpt_response','')).strip()).replace('\n',' ')
    for row in train_rows if row.get('prompt') and row.get('chatgpt_response')
) + '\n', encoding='utf-8')
EVAL_TEXT.write_text('\n'.join(
    (str(row.get('prompt','')).strip() + ' ' + str(row.get('chatgpt_response','')).strip()).replace('\n',' ')
    for row in eval_rows if row.get('prompt') and row.get('chatgpt_response')
) + '\n', encoding='utf-8')
print(f'Stable split: train={len(train_rows)}, independent holdout={len(eval_rows)}')
print('Calibration uses training documents only:', ANALYSIS_TEXT)
print('Teacher logits will be captured for training subset only:', INPUT_JSONL)

# Step 1: analyze actual FFN weights and measured gate/up activations.
subprocess.run([
    sys.executable, str(REPO / 'scripts/analyze_expert_channels.py'),
    '--model', MODEL_ID, '--text-file', str(ANALYSIS_TEXT),
    '--output', str(EXPERT_PLAN), '--num-experts', '32',
    '--samples', str(min(256, len(train_rows))), '--max-length', str(MAX_SEQ_LENGTH), '--device', 'cuda'
], check=True)
plan = json.loads(EXPERT_PLAN.read_text(encoding='utf-8'))
TEACHER_REVISION = plan['teacher_revision']
print('Pinned Qwen revision:', TEACHER_REVISION)

# Step 2: initialize checkpoints from the analysis-derived groups; no separate training run.
subprocess.run([
    sys.executable, str(REPO / 'scripts/build_moe_initial_checkpoints.py'),
    '--model', MODEL_ID, '--plan', str(EXPERT_PLAN),
    '--output-dir', str(INITIAL_CHECKPOINTS), '--num-experts', '32', '--top-k', '2'
], check=True)
print('Analysis-derived initial checkpoints:', INITIAL_CHECKPOINTS)


## قالب دقیق آرشیو متن

فایل `data/chatgpt_prompt_response_archive_part2.jsonl` در همین ریپو قرار دارد و هر خط آن یک JSON object مستقل است و حداقل این فیلدها را داشته باشد:

- `sample_id`: شناسهٔ یکتا و پایدار
- `prompt`: پرسش/دستور ورودی
- `chatgpt_response`: پاسخ کامل، باکیفیت و آمادهٔ آموزش

نمونهٔ زیر فقط قالب را نشان می‌دهد و خودکار به فایل اضافه نمی‌شود. متن واقعی را از ایجنت تولیدکننده وارد کن.

In [ ]:
example = {
    "sample_id": "fa-reasoning-000001",
    "prompt": "چرا تقسیم بر صفر در حساب معمول تعریف نشده است؟",
    "chatgpt_response": "در تقسیم a بر b، عددی می‌خواهیم که ضرب آن در b برابر a شود. وقتی b صفر باشد، برای a غیرصفر چنین عددی وجود ندارد؛ و برای a برابر صفر نیز جواب یکتا نیست."
}
print(json.dumps(example, ensure_ascii=False, indent=2))
print('Example only; it is not appended to the training archive.')

## مرحلهٔ ۱ — استخراج logits معلم

اسکریپت توالی دقیق prompt+answer، توکن‌های هدف، logits برتر Qwen و log-normalizer کل واژگان را ذخیره می‌کند. log-normalizer برای حفظ جرم احتمالاتی خارج از top-k استفاده می‌شود. این KD یک تقریب با tail تجمیع‌شده است، نه KL دقیق روی تمام واژگان.

خروجی‌ها روی Drive ذخیره می‌شوند و اجرای دوباره فایل‌های tensor موجود را رد می‌کند. ورودی متن مستقیماً از فایل داخل ریپوی GitHub خوانده می‌شود و از آرشیو Google Drive استفاده نمی‌شود.

In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'scripts/capture_qwen_logits_for_chatgpt_data.py'),
    '--input-jsonl', str(INPUT_JSONL),
    '--output-dir', str(LOGITS_OUT),
    '--model', MODEL_ID,
    '--revision', TEACHER_REVISION,
    '--top-k', str(TOP_K),
    '--temperature', str(TEMPERATURE),
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--device', 'auto',
], check=True)

INDEX_JSONL = LOGITS_OUT / 'qwen_logits_index.jsonl'
MANIFEST_JSON = LOGITS_OUT / 'qwen_logits_manifest.json'
print('Index:', INDEX_JSONL)
print('Manifest:', MANIFEST_JSON)
print(json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))) 

## مرحلهٔ ۲ — آموزش ترکیبی واحد

آموزش از ۲۴ checkpoint اولیهٔ MoE شروع می‌شود. مدل پایه با همان revision ثابت بارگذاری می‌شود، FFNهای dense با checkpointهای MoE جایگزین می‌شوند، وزن‌های backbone فریز می‌شوند و فقط expertها و routerها آموزش می‌بینند.

تابع هدف واحد است:

`total_loss = text_cross_entropy + 0.5 × qwen_sparse_KD`

Loss تقطیر از logits ذخیره‌شده استفاده می‌کند و جرم توکن‌های خارج از top-k را به‌صورت یک bucket تجمیع می‌کند. این نوت‌بوک هیچ اجرای text-only یا logits-only ندارد. تنظیمات پیش‌فرض یک epoch است؛ خروجی و وضعیت optimizer روی Drive ذخیره می‌شوند.

In [ ]:
TRAIN_OUT.mkdir(parents=True, exist_ok=True)

# پیش از حذف نسخهٔ قبلی، خروجی‌های لازم برای شروع آموزش جدید را اعتبارسنجی کن.
new_checkpoints = sorted(INITIAL_CHECKPOINTS.glob('layer_*_top2_32experts.pt'))
if len(new_checkpoints) != 24:
    raise RuntimeError(f'Expected 24 analysis-initialized checkpoints; found {len(new_checkpoints)}. Old checkpoints were preserved.')
# Validate the newly initialized set structurally before removing any old training checkpoints.
INITIAL_VALIDATION = ANALYSIS_DIR / 'initial_checkpoint_validation.json'
subprocess.run([
    sys.executable, str(REPO / 'scripts/validate_checkpoint_set.py'),
    '--checkpoint-dir', str(INITIAL_CHECKPOINTS), '--layers', '24',
    '--num-experts', '32', '--top-k', '2', '--output', str(INITIAL_VALIDATION)
], check=True)
initial_validation = json.loads(INITIAL_VALIDATION.read_text(encoding='utf-8'))
if initial_validation.get('status') != 'passed':
    raise RuntimeError('Initial checkpoint structural validation failed; old checkpoints were preserved.')
if not INDEX_JSONL.is_file() or not MANIFEST_JSON.is_file():
    raise RuntimeError('Teacher logits index/manifest missing; old checkpoints were preserved.')
logits_manifest = json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))
if logits_manifest.get('model_id') != MODEL_ID or logits_manifest.get('model_revision') != TEACHER_REVISION:
    raise RuntimeError('Teacher logits manifest model/revision mismatch; old checkpoints were preserved.')

# Kaggle execution is isolated: do not delete or overwrite any external/older checkpoint files.
train_cmd = [
    sys.executable, str(REPO / 'scripts/train_moe_chatgpt_qwen_kd.py'),
    '--index-jsonl', str(INDEX_JSONL),
    '--checkpoint-dir', str(INITIAL_CHECKPOINTS),
    '--output-dir', str(TRAIN_OUT),
    '--model', MODEL_ID,
    '--revision', TEACHER_REVISION,
    '--num-experts', '32',
    '--top-k', '2',
    '--epochs', '1',
    '--learning-rate', '1e-5',
    '--kd-weight', '0.5',
    '--temperature', str(TEMPERATURE),
    '--save-every', '200',
    '--device', 'auto',
]
# If this run was interrupted, continue from the saved MoE checkpoint/optimizer state.
if (TRAIN_OUT / 'training_state.pt').is_file() and (TRAIN_OUT / 'checkpoints').is_dir():
    train_cmd.append('--resume')
print('Running the single combined training route.')
subprocess.run(train_cmd, check=True)

## مرحلهٔ ۳ — اعتبارسنجی ساختاری checkpoint نهایی

این مرحله فقط کامل‌بودن و سازگاری ساختاری ۲۴ checkpoint را بررسی می‌کند؛ ادعای بهبود کیفیت یا آمادگی استقرار نیست. برای ارزیابی کیفیت باید مجموعهٔ ارزیابی مستقل و از قبل کنارگذاشته‌شده استفاده شود؛ دادهٔ آموزش نباید به‌عنوان ارزیابی گزارش شود.

In [ ]:
VALIDATION_JSON = TRAIN_OUT / 'checkpoint_validation.json'
subprocess.run([
    sys.executable, str(REPO / 'scripts/validate_checkpoint_set.py'),
    '--checkpoint-dir', str(TRAIN_OUT / 'checkpoints'),
    '--layers', '24',
    '--num-experts', '32',
    '--top-k', '2',
    '--output', str(VALIDATION_JSON),
], check=True)
print(VALIDATION_JSON.read_text(encoding='utf-8'))
print('Training summary:')
print((TRAIN_OUT / 'training_summary.json').read_text(encoding='utf-8'))

## منبع داده و فایل‌های خروجی

- ورودی ثابت از GitHub: `data/chatgpt_prompt_response_archive_part2.jsonl` (۱٬۲۷۵ نمونه)\n- `qwen_logits_output_v2/qwen_logits_index.jsonl` و پوشهٔ `qwen_topk_logits/`: سیگنال عددی معلم
- `combined_moe_training_v2/checkpoints/`: checkpointهای نهایی آموزش‌شده
- `combined_moe_training_v2/training_metrics.jsonl`: Loss و گرادیان هر مرحله
- `combined_moe_training_v2/training_state.pt`: وضعیت optimizer برای ادامهٔ آموزش
- `combined_moe_training_v2/training_summary.json`: خلاصهٔ اجرا

محدودیت‌ها: این نوت‌بوک و کد باید روی GPU واقعی اجرا شوند تا اجرای موفق تأیید شود. تقطیر top-k با tail تجمیع‌شده تقریب KL است. این روش به‌تنهایی تضمین کیفیت پاسخ یا اجرای سریع روی Android نیست.

In [ ]:
# End-to-end comparison on documents never used for activation analysis, logits training, or KD.
EVAL_REPORT = TRAIN_OUT / 'evaluation_holdout.json'
subprocess.run([
    sys.executable, str(REPO / 'scripts/evaluate_full_model.py'),
    '--model', MODEL_ID, '--checkpoint-dir', str(TRAIN_OUT / 'checkpoints'),
    '--text-file', str(EVAL_TEXT), '--output', str(EVAL_REPORT),
    '--max-documents', str(len(eval_rows)), '--max-length', str(MAX_SEQ_LENGTH),
    '--num-experts', '32', '--top-k', '2', '--device', 'cuda'
], check=True)
eval_report = json.loads(EVAL_REPORT.read_text(encoding='utf-8'))
print(json.dumps(eval_report, ensure_ascii=False, indent=2))

# Write a compact machine-readable result for automatic GitHub synchronization.
RESULTS_JSON = Path('/kaggle/working/qwen_moe_results.json')
summary_path = TRAIN_OUT / 'training_summary.json'
validation_path = TRAIN_OUT / 'checkpoint_validation.json'
results_payload = {
    'status': 'completed',
    'model_id': MODEL_ID,
    'teacher_revision': str(TEACHER_REVISION),
    'gpu': __import__('torch').cuda.get_device_name(0),
    'dataset': {
        'total_examples': len(all_rows),
        'train_examples': len(train_rows),
        'holdout_examples': len(eval_rows),
        'max_seq_length': MAX_SEQ_LENGTH,
    },
    'training_summary': json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.exists() else None,
    'checkpoint_validation': json.loads(validation_path.read_text(encoding='utf-8')) if validation_path.exists() else None,
    'evaluation_holdout': eval_report,
    'quality_guardrails': {
        'perplexity_ratio_limit': 1.15,
        'next_token_agreement_minimum': 0.80,
        'perplexity_ratio': eval_report.get('perplexity_ratio_converted_over_teacher'),
        'next_token_agreement': eval_report.get('converted', {}).get('next_token_agreement'),
        'passed': (
            eval_report.get('perplexity_ratio_converted_over_teacher') is not None
            and eval_report.get('perplexity_ratio_converted_over_teacher') <= 1.15
            and eval_report.get('converted', {}).get('next_token_agreement') is not None
            and eval_report.get('converted', {}).get('next_token_agreement') >= 0.80
        ),
    },
}
RESULTS_JSON.write_text(json.dumps(results_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Compact GitHub-sync result:', RESULTS_JSON)
print(json.dumps(results_payload['quality_guardrails'], ensure_ascii=False, indent=2))

# Package only when the structural/end-to-end guardrails pass; this is still an overlay, not a standalone Android model.
ratio = eval_report.get('perplexity_ratio_converted_over_teacher')
agreement = eval_report.get('converted', {}).get('next_token_agreement')
if ratio is not None and ratio <= 1.15 and agreement is not None and agreement >= 0.80:
    PACKAGE_OUT = WORK / 'qwen_moe_overlay'
    subprocess.run([
        sys.executable, str(REPO / 'scripts/package_moe_adapter.py'),
        '--checkpoint-dir', str(TRAIN_OUT / 'checkpoints'),
        '--output-dir', str(PACKAGE_OUT), '--model', MODEL_ID,
        '--model-revision', TEACHER_REVISION, '--layers', '24',
        '--num-experts', '32', '--top-k', '2'
    ], check=True)
    print('Guardrails passed. Experimental overlay package:', PACKAGE_OUT)
else:
    print('PACKAGE SKIPPED: holdout perplexity ratio must be <= 1.15 and next-token agreement >= 0.80.')
    print('This is a quality guardrail, not a comprehensive Persian reasoning benchmark. No package was produced.')


# Export compact, reproducible artifacts as a Kaggle notebook output.
ARCHIVE = Path('/kaggle/working/qwen_moe_run_artifacts.zip')
include_paths = [
    TRAIN_OUT / 'checkpoints',
    TRAIN_OUT / 'checkpoint_validation.json',
    TRAIN_OUT / 'training_summary.json',
    TRAIN_OUT / 'evaluation_holdout.json',
    TRAIN_OUT / 'training_state.pt',
    ANALYSIS_DIR / 'expert_channel_plan.json',
    ANALYSIS_DIR / 'initial_checkpoint_validation.json',
    LOGITS_OUT / 'qwen_logits_manifest.json',
    LOGITS_OUT / 'qwen_logits_index.jsonl',
]
with zipfile.ZipFile(ARCHIVE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for item in include_paths:
        if item.is_file():
            zf.write(item, item.relative_to(WORK))
        elif item.is_dir():
            for p in item.rglob('*'):
                if p.is_file():
                    zf.write(p, p.relative_to(WORK))
print('Kaggle output archive:', ARCHIVE, 'bytes=', ARCHIVE.stat().st_size)
print('Output directory:', WORK)


## بازیابی خروجی‌ها

پس از اتمام اجرا، فایل qwen_moe_run_artifacts.zip و پوشهٔ خروجی Notebook را از تب Output این Kernel دریافت کن. اجرای موفق این نوت‌بوک به معنی قبولی معیارهای کیفیت نیست؛ نتیجهٔ ارزیابی holdout و وضعیت بسته‌بندی را بررسی کن. فایل‌های قبلی خارج از این محیط حذف نمی‌شوند.